In [1]:
import pandas as pd
import requests

In [2]:
url = 'https://servicios.ine.es/wstempus/js/es/DATOS_TABLA/31097?tip=A&tv=846:'
api_distritos = requests.get(url)
api_distritos

<Response [200]>

In [3]:
datos_json = api_distritos.json()
df = pd.DataFrame(datos_json)
df.head()

,COD,Nombre,T3_Unidad,T3_Escala,Data
0,ADRH83527,"Acebeda, La distrito 01. Dato base. Renta neta...",Euros,,"[{'Fecha': '2023-01-01T00:00:00.000+01:00', 'T..."
1,ADRH83526,"Acebeda, La distrito 01. Dato base. Renta neta...",Euros,,"[{'Fecha': '2023-01-01T00:00:00.000+01:00', 'T..."
2,ADRH9811731,"Acebeda, La distrito 01. Dato base. Media de l...",Euros,,"[{'Fecha': '2023-01-01T00:00:00.000+01:00', 'T..."
3,ADRH9552121,"Acebeda, La distrito 01. Dato base. Mediana de...",Euros,,"[{'Fecha': '2023-01-01T00:00:00.000+01:00', 'T..."
4,ADRH9552119,"Acebeda, La distrito 01. Dato base. Renta brut...",Porcentaje,,"[{'Fecha': '2023-01-01T00:00:00.000+01:00', 'T..."


In [4]:
#DATA tiene la informacion de fecha como del valor de renta, asi que hay que expandirla
df_expandido = df.explode("Data").copy()
import json

def extraer_campo(x, campo):
    if isinstance(x, str):
        try:
            x = json.loads(x.replace("'", '"'))
        except:
            return None
    if isinstance(x, dict):
        return x.get(campo)
    return None


In [5]:
df_expandido["Fecha"] = df_expandido["Data"].apply(lambda x: extraer_campo(x, "Anyo"))
df_expandido["Valor"] = df_expandido["Data"].apply(lambda x: extraer_campo(x, "Valor"))
df_expandido.head()

,COD,Nombre,T3_Unidad,T3_Escala,Data,Fecha,Valor
0,ADRH83527,"Acebeda, La distrito 01. Dato base. Renta neta...",Euros,,"{'Fecha': '2023-01-01T00:00:00.000+01:00', 'T3...",2023,16893.0
0,ADRH83527,"Acebeda, La distrito 01. Dato base. Renta neta...",Euros,,"{'Fecha': '2022-01-01T00:00:00.000+01:00', 'T3...",2022,15715.0
0,ADRH83527,"Acebeda, La distrito 01. Dato base. Renta neta...",Euros,,"{'Fecha': '2021-01-01T00:00:00.000+01:00', 'T3...",2021,15245.0
0,ADRH83527,"Acebeda, La distrito 01. Dato base. Renta neta...",Euros,,"{'Fecha': '2020-01-01T00:00:00.000+01:00', 'T3...",2020,13999.0
0,ADRH83527,"Acebeda, La distrito 01. Dato base. Renta neta...",Euros,,"{'Fecha': '2019-01-01T00:00:00.000+01:00', 'T3...",2019,NaN


In [6]:
filtro_madrid = df_expandido["Nombre"].str.contains("^Madrid", case=False, na=False)
filtro_fechas = df_expandido["Fecha"].isin(list(range(2015, 2023)))
filtro_T3unidad = df_expandido["T3_Unidad"].str.strip() == "Euros"
df_madrid_filtrado = df_expandido[filtro_madrid & filtro_fechas & filtro_T3unidad].copy()

In [7]:
separacion_nombre = df_madrid_filtrado["Nombre"].str.split(".", n=1, expand=True)
df_madrid_filtrado["Distrito madrid"] = separacion_nombre[0].str.strip()
df_madrid_filtrado["Dato base"] = separacion_nombre[1].str.strip()

In [8]:
indicadores_datobase = ["Dato base. Renta neta media por persona.","Dato base. Renta neta media por hogar."]
filtro_rentas = df_madrid_filtrado["Dato base"].isin(indicadores_datobase)
df_madrid_filtrado = df_madrid_filtrado[filtro_rentas]
df_madrid_filtrado['Distrito madrid'] = df_madrid_filtrado['Distrito madrid'].str[-2:]

In [9]:
columnas_drop = ["Data", "T3_Escala", "T3_Unidad", "Nombre"]
df_madrid_distritos = df_madrid_filtrado.drop(columns=columnas_drop).rename(columns={"Fecha": "Año"})
df_madrid_distritos

,COD,Año,Valor,Distrito madrid,Dato base
666,ADRH84859,2022,20587.0,01,Dato base. Renta neta media por persona.
666,ADRH84859,2021,19199.0,01,Dato base. Renta neta media por persona.
666,ADRH84859,2020,18314.0,01,Dato base. Renta neta media por persona.
666,ADRH84859,2019,18789.0,01,Dato base. Renta neta media por persona.
666,ADRH84859,2018,17932.0,01,Dato base. Renta neta media por persona.
...,...,...,...,...,...
787,ADRH68150,2019,51108.0,21,Dato base. Renta neta media por hogar.
787,ADRH68150,2018,49615.0,21,Dato base. Renta neta media por hogar.
787,ADRH68150,2017,47782.0,21,Dato base. Renta neta media por hogar.
787,ADRH68150,2016,47389.0,21,Dato base. Renta neta media por hogar.


In [10]:
df_madrid_distritos["tipo"] = df_madrid_distritos["Dato base"].str.contains("hogar").map({True: "hogar", False: "persona"})
df_madrid_distritos = df_madrid_distritos.drop(columns=['COD', 'Dato base'])
df_madrid_distritos = df_madrid_distritos.rename(columns={'Distrito madrid': 'COD_DIS_MADRID'})
df_madrid_distritos['COD_DIS_MADRID'] = "28079" + df_madrid_distritos['COD_DIS_MADRID'].astype(str)

In [11]:
print(df_madrid_distritos["Año"].isna().sum())      
print(df_madrid_distritos["COD_DIS_MADRID"].nunique()) 
df_madrid_distritos.head(40)

0
21


,Año,Valor,COD_DIS_MADRID,tipo
666,2022,20587.0,2807901,persona
666,2021,19199.0,2807901,persona
666,2020,18314.0,2807901,persona
666,2019,18789.0,2807901,persona
666,2018,17932.0,2807901,persona
666,2017,16711.0,2807901,persona
666,2016,16147.0,2807901,persona
666,2015,15512.0,2807901,persona
667,2022,41059.0,2807901,hogar
667,2021,38360.0,2807901,hogar
